# SIC Model Inference, Evaluation, and Visualization

**Main Components:**
* **Response Prediction**: Neural dynamics simulation using the SIC model.
* **Metric Extraction**: Signal alignment and quantitative comparison (SIC vs. DMN).
* **Data Visualization**: Statistical plotting.

## 1. Environment Setup and Dependency Loading
This block configures the system path to include local utility modules and imports the core SIC (Signal Infinite Cascade) model components.

In [1]:
import sys
import os

# Add the 'util' directory to the system path to allow importing local modules
sys.path.append(os.path.abspath('../util'))

# Import custom stimulus processing and SIC modeling classes
from stimulus_Loader import StimulusProcessor
from SIC import SICModelTorch

## 2. SIC Model Inference Pipeline
This module executes the **SIC (Signal Infinite Cascade) model** to compute neural response dynamics. 

### Core Scientific Objective:
To transform visual stimuli into predicted neural activity. This is achieved by:

In [ ]:
import numpy as np
import csv
from tqdm import tqdm
from datetime import datetime
import torch

# --- Load Stimuli from CSV File into Memory ---
print("Reading stimuli from CSV file...")

stimulus_dataset = {}
csv_file = '../refer/onoff-mask.csv'

with open(csv_file, 'r') as f:
    reader = csv.reader(f)
    rows = list(reader)

stim_names = [name.strip() for name in rows[0][1:] if name.strip() != '']

# Parse intensity values to float32; map empty strings to NaN for temporal alignment
data_rows = rows[1:]
data_array = np.array([[float(x) if x.strip() != '' else np.nan for x in row[1:len(stim_names)+1]] 
                       for row in data_rows], dtype=np.float32)

# --- Convert 1D Temporal Signals to 3D Spatio-temporal Tensors ---
PRE_STEPS = 600
STEP_SIZE = 10  
n_stimuli = len(stim_names)

for idx, stim_name in enumerate(stim_names):
    col_data = data_array[:, idx]
    valid_mask = col_data[~np.isnan(col_data)]
    if len(valid_mask) == 0:
        print(f"Warning: stimulus {stim_name} has no valid values, skipping.")
        continue
    n_time = len(valid_mask)
    stimulus = np.zeros((41, 82, PRE_STEPS + n_time * STEP_SIZE), dtype=np.float32)
    
    stimulus[:, :, :PRE_STEPS] = valid_mask[0]
    
    for t, val in enumerate(valid_mask):
        start = PRE_STEPS + t * STEP_SIZE
        end = start + STEP_SIZE
        stimulus[:, :, start:end] = val
    
    stimulus_dataset[stim_name] = stimulus

print(f"✅ Total stimuli loaded: {len(stimulus_dataset)}")

# --- Initialize SIC Model and Load Connectivity Matrices ---
SIC_model = SICModelTorch(
    matrix_dir="neuron_matrices",
    output_dir="../results/responses/fri",
    t_step=10,
    rate=100,
    device=torch.device("cuda:0")
)

def read_all_neuron_ids(filename='../data/classification.txt'):
    neuron_ids = []
    try:
        with open(filename, 'r') as f:
            reader = csv.DictReader(f)
            for row in reader:
                if 'root_id' in row and row['root_id'].strip():
                    try:
                        neuron_ids.append(int(row['root_id'].strip()))
                    except ValueError:
                        continue
    except FileNotFoundError:
        print(f"Error: File '{filename}' not found")
    except Exception as e:
        print(f"Error reading file: {str(e)}")
    return neuron_ids

neuron_ids = read_all_neuron_ids()
print(f"Found {len(neuron_ids)} neurons")

weights = SIC_model.load_weights(neuron_ids, normalize=False)
print("Weights loaded successfully")

# --- Iterate Through Stimuli and Execute Forward Pass ---
print("Calculating SIC responses...")
for stim_name, stimulus in tqdm(
    stimulus_dataset.items(),
    desc="Processing stimuli",
    unit="stimulus"
):
    SIC_model.calculate_response_baseline(
        stimulus,
        weights,
        responce_threshold=0,
        baseline_steps=50,
        stim_name=stim_name
    )

print("\n" + "="*50)
print("PREPROCESSING COMPLETE")
print("="*50)
print(f"Total stimuli processed: {len(stimulus_dataset)}")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*50)

Reading stimuli from CSV file...
✅ Total stimuli loaded: 34
Using device: cuda:0
Found 139255 neurons


## 3. Visualization Style Configuration

In [ ]:
import matplotlib.pyplot as plt
import scienceplots

plt.style.use(['science', 'nature', 'no-latex'])

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans"],  # Ubuntu 推荐
    "font.size": 16,
    "axes.labelsize": 16,
    "axes.titlesize": 20,
    "legend.fontsize": 16,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "axes.linewidth": 1.5,
})

## 4. Quantitative Model Evaluation
This module performs a comparative analysis between the predicted neural responses (SIC and DMN models) and the experimental ground truth (`Exp_Value`).

### Key Analytical Features:
- **Temporal Alignment**: Computes the optimal time-lag between signals via cross-correlation to account for systematic delays.
- **Performance Metrics**: 
    - **Pearson Correlation ($r$)**: Measures linear similarity.
    - **Mean Squared Error (MSE)**: Quantifies absolute magnitude deviation.
    - **Dynamic Time Warping (DTW)**: Assesses structural similarity for non-linear temporal sequences.
- **Batch Processing**: Recursively aggregates metrics from all result CSVs into a summary file (`evaluation_metrics.csv`).

In [ ]:
import pandas as pd
import scipy.stats as stats
import numpy as np
import os
from tqdm import tqdm
try:
    from dtaidistance import dtw
except ImportError:
    dtw = None

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  
base_dir = "../results/Figure3b"
output_file = "../refer/evaluation_metrics.csv"

results_dict = {}
target_headers = {'Time (s)', 'Mask', 'Exp_Value', 'SIC_Value', 'DMN_Value'}

# --- 1. Metric Calculation Engine ---
def get_aligned_metrics(target_array, exp_array):
    target_array = np.asarray(target_array, dtype=float).flatten()
    exp_array = np.asarray(exp_array, dtype=float).flatten()

    if len(target_array) < 2 or len(exp_array) < 2:
        return None, None, None, None

    # Compute best lag to align model output with experimental recording
    t_centered = target_array - np.mean(target_array)
    e_centered = exp_array - np.mean(exp_array)
    corr = np.correlate(e_centered, t_centered, mode='full')
    lags = np.arange(-len(t_centered) + 1, len(e_centered))
    best_lag = lags[np.argmax(corr)]

    # Align arrays based on the identified best lag
    if best_lag > 0:
        a_exp = exp_array[best_lag:]
        a_target = target_array[:len(target_array)-best_lag]
    elif best_lag < 0:
        a_exp = exp_array[:len(exp_array)+best_lag]
        a_target = target_array[-best_lag:]
    else:
        a_exp = exp_array
        a_target = target_array

    if len(a_exp) < 2:
        return best_lag, None, None, None

    # Compute correlation, MSE, and DTW
    r, _ = stats.pearsonr(a_exp, a_target)
    mse = np.mean((a_target - a_exp) ** 2)

    dtw_score = None
    if dtw is not None:
        dtw_score = dtw.distance(a_exp, a_target) / len(a_exp)

    return best_lag, r, mse, dtw_score

# --- 2. Batch Evaluation Loop ---
for root, dirs, files in os.walk(base_dir):
    for file in files:
        if not file.endswith(".csv"):
            continue

        file_path = os.path.join(root, file)

        try:
            df_head = pd.read_csv(file_path, nrows=0)
            # Ensure numeric conversion for evaluation
            actual_headers = set(df_head.columns.str.strip())

            if not target_headers.issubset(actual_headers):
                continue

            df = pd.read_csv(file_path)

            for col in ['Exp_Value', 'SIC_Value', 'DMN_Value']:
                df[col] = pd.to_numeric(df[col], errors='coerce')

            df_sic = df.dropna(subset=['Exp_Value', 'SIC_Value'])
            df_dmn = df.dropna(subset=['Exp_Value', 'DMN_Value'])

            # --- SIC vs Exp ---
            if len(df_sic) >= 2 and np.std(df_sic['Exp_Value']) > 0:
                sic_lag, sic_cc, sic_mse, sic_dtw = get_aligned_metrics(
                    df_sic['Exp_Value'].values, df_sic['SIC_Value'].values
                )
            else:
                sic_lag, sic_cc, sic_mse, sic_dtw = None, None, None, None

            # --- DMN vs Exp ---
            if len(df_dmn) >= 2 and np.std(df_dmn['Exp_Value']) > 0:
                dmn_lag, dmn_cc, dmn_mse, dmn_dtw = get_aligned_metrics(
                    df_dmn['Exp_Value'].values, df_dmn['DMN_Value'].values
                )
            else:
                dmn_lag, dmn_cc, dmn_mse, dmn_dtw = None, None, None, None

            label = file.replace("_extracted.csv", "").replace(".csv", "")
            results_dict[label] = {
                "SIC_Best Lag": sic_lag,
                "SIC_MCC": sic_cc, 
                "SIC_MSE": sic_mse,
                "SIC_DTW": sic_dtw,
                "DMN_Best Lag": dmn_lag,
                "DMN_MCC": dmn_cc, 
                "DMN_MSE": dmn_mse,
                "DMN_DTW": dmn_dtw
            }

        except Exception as e:
            print(f"⚠️{e}")

# --- 3. Export Summary Statistics ---
if results_dict:
    results_df = pd.DataFrame(results_dict)
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    results_df.to_csv(output_file, float_format="%.4f")


## 5. Visualization of Model Performance
This module generates comparative figures to evaluate the SIC and DMN models based on the metrics derived in the previous section.

### Figure Design:
- **Grouped Bar Charts**: Displays performance metrics (Best Lag, MCC, MSE, DTW) side-by-side for SIC and DMN models.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import scienceplots

plt.style.use(['science', 'nature', 'no-latex'])
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans"],
    "font.size": 20,
    "axes.labelsize": 24,
    "axes.titlesize": 24,
    "legend.fontsize": 18,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "axes.linewidth": 1.5,
})

# --- 1. Data Loading and Formatting ---
file_path = "../refer/evaluation_metrics.csv"
df_raw = pd.read_csv(file_path)
df_raw.rename(columns={df_raw.columns[0]: "Metric"}, inplace=True)

# Separate performance data for each model using string masks
sic_mask = df_raw["Metric"].str.startswith("SIC")
dmn_mask = df_raw["Metric"].str.startswith("DMN")

sic_values = df_raw[sic_mask].iloc[:, 1:].astype(float).values
dmn_values = df_raw[dmn_mask].iloc[:, 1:].astype(float).values

sic_metrics = [m.replace("SIC_", "") for m in df_raw[sic_mask]["Metric"].tolist()]

n_models = sic_values.shape[1]

deep_blue = '#1E3A8A'   # SIC
deep_red = '#8B0000'    # DMN

# --- 2. Plotting Engine ---
def plot_metric_subplot(ax, metric_idx, metric_name, sic_vals, dmn_vals):
    x = np.arange(1, sic_vals.shape[1]+1)  
    width = 0.35

    sic_data = sic_vals[metric_idx, :]
    dmn_data = dmn_vals[metric_idx, :]

    ax.bar(x - width/2, sic_data, width, label="SIC", color=deep_blue)
    ax.bar(x + width/2, dmn_data, width, label="DMN", color=deep_red)

    ax.set_xticks(x)
    ax.set_xticklabels(x, rotation=90)
    ax.set_ylabel("Value")
    ax.set_title(metric_name)

# --- 3. Generate Multi-panel Figure ---
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes = axes.flatten()

for i, metric in enumerate(sic_metrics):
    plot_metric_subplot(axes[i], i, metric, sic_values, dmn_values)

axes[0].legend(loc='lower right')

plt.tight_layout()
plt.show()

## 6. Statistical Analysis & Comparative Visualization
This final module aggregates the performance metrics across all test samples to provide a statistical comparison between the **SIC** and **DMN** models.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import scienceplots

# --- 1. Global Visualization Settings ---
plt.style.use(['science', 'nature', 'no-latex'])
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans"],
    "font.size": 40,
    "axes.labelsize": 40,
    "axes.titlesize": 40,
    "legend.fontsize": 40,
    "xtick.labelsize": 40,
    "ytick.labelsize": 40,
    "axes.linewidth": 1.5,
})

# --- 2. Data Ingestion & Preprocessing ---
file_path = "../refer/evaluation_metrics.csv"
df_raw = pd.read_csv(file_path)
df_raw.rename(columns={df_raw.columns[0]: "Metric"}, inplace=True)

# Group metrics by model prefix (SIC vs DMN)
sic_mask = df_raw["Metric"].str.startswith("SIC")
dmn_mask = df_raw["Metric"].str.startswith("DMN")

sic_values = df_raw[sic_mask].iloc[:, 1:].astype(float).values
dmn_values = df_raw[dmn_mask].iloc[:, 1:].astype(float).values

sic_metrics = [m.replace("SIC_", "") for m in df_raw[sic_mask]["Metric"].tolist()]

# Define color palette for model distinction
deep_blue = '#1E3A8A'   # SIC
deep_red = '#8B0000'    # DMN

# --- 3. Plotting Function Definitions ---
def plot_metric_subplot(ax, metric_idx, metric_name, sic_vals, dmn_vals, left_label=None):
    sic_data = sic_vals[metric_idx, :].copy()
    dmn_data = dmn_vals[metric_idx, :].copy()

    # Unit conversion: Best Lag -> ms
    if metric_name == "Best Lag":
        sic_data *= 10
        dmn_data *= 10

    sic_mean = np.nanmean(sic_data)
    sic_std = np.nanstd(sic_data)
    dmn_mean = np.nanmean(dmn_data)
    dmn_std = np.nanstd(dmn_data)

    x = np.arange(2)  # 0=SIC, 1=DMN
    means = [sic_mean, dmn_mean]
    errors = [sic_std, dmn_std]
    labels = ["SIC", "DMN"]
    colors = [deep_blue, deep_red]

    # Render bars
    ax.bar(x, means, yerr=errors, capsize=10, color=colors)

    # Configure axes
    ax.set_xticks(x)
    ax.set_xticklabels(labels)

    if left_label is not None:
        if metric_name.lower() == "best lag":
            arrow = "→←"  
        elif metric_name.lower() == "MCC":
            left_label = 'MCC'
            arrow = "→" 
        else:
            arrow = "←"

        ax.set_ylabel(f"{left_label} ({arrow})", rotation=90, labelpad=20, va='center')
    else:
        ax.set_ylabel("")

    ax.set_title("")

# --- 4. Main Visualization Loop ---
fig, axes = plt.subplots(1, 4, figsize=(30, 10))
axes = axes.flatten()

for i, metric in enumerate(sic_metrics):
    plot_metric_subplot(axes[i], i, metric, sic_values, dmn_values, left_label=metric)

plt.subplots_adjust(wspace=0.5)
plt.show()

# --- Logging Summary Statistics to Console ---
print("Metric\tSIC mean\tSIC std\tDMN mean\tDMN std")
for i, metric in enumerate(sic_metrics):
    sic_data = sic_values[i, :].copy()
    dmn_data = dmn_values[i, :].copy()

    if metric == "Best Lag":
        sic_data *= 10
        dmn_data *= 10

    sic_mean = np.nanmean(sic_data)
    sic_std = np.nanstd(sic_data)
    dmn_mean = np.nanmean(dmn_data)
    dmn_std = np.nanstd(dmn_data)

    print(f"{metric}\t{sic_mean:.3f}\t{sic_std:.3f}\t{dmn_mean:.3f}\t{dmn_std:.3f}")